In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Paths to results
BASE_DIR = Path("results/mmlu_random_vectors")
MODEL_DIRS = {
    "Qwen2.5-7B": BASE_DIR / "Qwen_Qwen2.5_7B_Instruct" / "mmlu_random_vectors_results.json",
    "Falcon3-7B": BASE_DIR / "tiiuae_Falcon3_7B_Instruct" / "mmlu_random_vectors_results.json",
    "Llama-3.1-8B": BASE_DIR / "meta_llama_Llama_3.1_8B_Instruct" / "mmlu_random_vectors_results.json",
}

MODEL_DIRS['Llama-3.1-8B-SAE'] = Path("results/mmlu_random_sae_features/meta_llama_Llama_3.1_8B_Instruct/mmlu_random_sae_features_results.json")

MODEL_DIRS


In [ ]:
from scipy import stats

def load_results(path: str):
    with open(path, "r") as f:
        data = json.load(f)
    return data


def coef_to_stats(results: dict, ci_level=0.999, pct_level=0.99):
    """Return dict: relative_coef (float) -> dict with mean, ci_low, ci_high (for mean) and pct_low, pct_high (for distribution)."""
    rel_coefs = results["relative_coefficients"]
    steered = results["steered_accuracies"]
    baseline_overall = results["baseline"]["overall_accuracy"]

    # Start with baseline at coef 0
    coef_to_vals = {0.0: []}
    for c in rel_coefs:
        coef_to_vals[float(c)] = []

    for vec_key, vec_res in steered.items():
        # baseline: same for all vectors (but keep per-vector so median logic is consistent)
        coef_to_vals[0.0].append(baseline_overall)
        for c in rel_coefs:
            c_str = str(c)
            if c_str in vec_res and isinstance(vec_res[c_str], dict):
                coef_to_vals[float(c)].append(vec_res[c_str]["overall_accuracy"])

    # Compute mean, CI for mean (99%), and percentiles for distribution (75%)
    ci_alpha = 1 - ci_level
    pct_alpha = 1 - pct_level
    coef_stats = {}
    for c, vals in coef_to_vals.items():
        if len(vals) == 0:
            coef_stats[c] = {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "pct_low": np.nan, "pct_high": np.nan}
        else:
            n = len(vals)
            mean_val = float(np.mean(vals))
            std_val = float(np.std(vals, ddof=1)) if n > 1 else 0
            
            # Calculate 99% confidence interval for mean using t-distribution
            if n > 1:
                z = stats.t.ppf(1 - ci_alpha / 2, df=n-1)
                margin = z * std_val / np.sqrt(n)
                ci_low = max(0, mean_val - margin)
                ci_high = min(1, mean_val + margin)
            else:
                ci_low, ci_high = mean_val, mean_val
            
            # Calculate 75% percentiles for the full distribution (12.5th to 87.5th)
            pct_low = float(np.percentile(vals, pct_alpha / 2 * 100))
            pct_high = float(np.percentile(vals, (1 - pct_alpha / 2) * 100))
                
            coef_stats[c] = {
                "mean": mean_val, 
                "ci_low": ci_low, 
                "ci_high": ci_high,
                "pct_low": pct_low,
                "pct_high": pct_high
            }
    return coef_stats


def summarize_model(path: str):
    res = load_results(path)
    return coef_to_stats(res), res["model_name"], res["layer_index"], res["sample_size"]

summaries = {name: summarize_model(str(path)) for name, path in MODEL_DIRS.items()}

summaries.keys()


In [ ]:
# Define configuration for each model to match ablation.ipynb style exactly
from collections import OrderedDict
model_config = OrderedDict()

model_config = {
    "Qwen2.5-7B": {'color': "#4ECD96", 'marker': 's', 'label': 'Qwen2.5-7B layer 14 (1/2 depth, Random)', 'linewidth':'10'},
    "Falcon3-7B": {'color':'#45B7D1', 'marker': 's', 'label': 'Falcon3-7B layer 14 (1/2 depth, Random)', 'linewidth':'10'},
    "Llama-3.1-8B": {'color': '#e19327' , 'marker': 's', 'label': 'Llama3.1-8B layer 16 (1/2 depth, Random)', 'linewidth':'10'},
    "Llama-3.1-8B-SAE": {'color': "#FF6B6B" , 'marker': 'o', 'label': 'Llama3.1-8B layer 19 (2/3 depth, SAE)', 'linewidth':'4'},
}

# ============================================
# Plot 1: CI for the Mean (99%)
# ============================================
def plot_mmlu_ci(y_max=None):
    fig, ax = plt.subplots(figsize=(10, 7.5))
    plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.12)
    
    for short_name, (coef_stats, model_name, layer_idx, sample_size) in summaries.items():
        xs = sorted(coef_stats.keys())
        ys = np.array([coef_stats[x]["mean"] * 100 for x in xs])  # Convert to percentage
        ci_lows = np.array([coef_stats[x]["ci_low"] * 100 for x in xs])
        ci_highs = np.array([coef_stats[x]["ci_high"] * 100 for x in xs])
        
        # Prepare error bars
        y_err_lower = ys - ci_lows
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = ci_highs - ys
        y_err = [y_err_lower, y_err_upper]
        
        config = model_config[short_name]

        
        ax.errorbar(
            xs,
            ys,
            yerr=y_err,
            label=config['label'],
            color=config['color'],
            marker=config['marker'],
            capsize=5,
            linestyle='-',
            linewidth=config['linewidth'],
            markersize=13,
            alpha=1,
            ecolor='black', elinewidth=2, capthick=2,
        )

    ax.set_xlabel("Scaling Coefficient (Steering Strength)", fontsize=22)
    ax.set_ylabel("MMLU Accuracy (%)", fontsize=24)

    # Red dashed reference line + annotation
    # ax.axvline(0.6, color='red', linestyle='--', linewidth=3)
    # ax.text(
    #     0.02, 0.37,
    #     "Model\nCapabilities\nPreserved",
    #     transform=ax.transAxes,
    #     ha='left', va='bottom',
    #     fontsize=24,
    #     color='red',
    # )

    # ax.text(
    #     0.72, 0.37,
    #     "Model\nDegradation\nRegime",
    #     transform=ax.transAxes,
    #     ha='left', va='bottom',
    #     fontsize=24,
    #     color='red',
    # )

    ax.grid(True, alpha=1)

    ax.tick_params(axis='both', labelsize=18)
    ax.legend(fontsize=20, loc='upper right')
    
    if y_max is not None:
        ax.set_ylim(0, y_max)
    
    plt.show()

plot_mmlu_ci(y_max=100)
